# 10-key finite-key vs. asymptotic recompute (reviewer feedback #9)

Moved **verbatim** from cell 16 of `notebooks/core/30_scale_up_experiments.ipynb` (formerly B13 in `CLUTTER_REVIEW.md`) on 2026-10-27. It is real, intentional work answering reviewer feedback #9 (all 10 keys), but it is **not** what the paper's Table 5 reports: Table 5 uses the original 5 keys (see `results/data/key_pairs_metadata_table5_5keys.csv`, 93.76%).

* Output on disk: `results/data/finitekey_vs_asymptotic_table2_10keys.csv` (10-key mean reduction 96.05%; first 5 keys 93.76%).
* As it was in the source notebook, the cell is **not runnable alone**: it needs `combined` (10-row metadata DataFrame, built in `30_scale_up_experiments` cell 10) and `asymptotic_key_length`, `finite_key_output_length` (`from qne.cascade.finite_key import ...`, which lived in a second-site cell now in `second_site_ncsa_toky_experiments.ipynb`). It ended in a NameError there too.
* Not used by `master_reproduction.ipynb`.


In [14]:
# --- Recompute Table 2 (finite-key vs. asymptotic length) for all 10 keys ---
rows = []
for _, row in combined.iterrows():
    i, n, k, qber = int(row["index"]), int(row["n_bits"]), int(row["k_pe"]), float(row["qber"])
    ell_asymptotic = asymptotic_key_length(n, qber)
    try:
        ell_finite, r, t, nu = finite_key_output_length(n, k, qber)
    except ValueError as e:
        ell_finite = None
        print(f"key {i}: finite-key optimization failed ({e})")
    reduction = (1 - ell_finite / ell_asymptotic) * 100 if ell_finite and ell_asymptotic else None
    rows.append({"key": i, "n_bits": n, "qber": qber,
                 "ell_asymptotic": ell_asymptotic, "ell_finite": ell_finite, "reduction_pct": reduction})

table2_extended = pd.DataFrame(rows)
table2_extended.to_csv(str(RESULTS / "data" / "finitekey_vs_asymptotic_table2_10keys.csv"), index=False)
print(table2_extended.to_string(index=False))
print(f"\nOriginal 5-key mean reduction: {table2_extended.iloc[:5]['reduction_pct'].mean():.2f}%")
print(f"All-10-key mean reduction:     {table2_extended['reduction_pct'].mean():.2f}%")
print("(Exclude any key with an extreme ell_finite, e.g. <=1 bit, from the mean the same "
      "way key 1 was excluded in the original 5-key table.)")


NameError: name 'asymptotic_key_length' is not defined